## Problem 1

### Generating Random Boolean Functions
The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

In [ ]:
import random
from itertools import product

def random_constant_balanced():
    """
    Return a randomly chosen Boolean function f(a, b, c, d) that is either:
    - constant (all False or all True), or
    - balanced (True for exactly 8 of the 16 input combinations).
    """
    inputs = list(product([False, True], repeat=4))

    if random.choice(["constant", "balanced"]) == "constant":
        value = random.choice([False, True])

        def f(a, b, c, d):
            return value

    else:
        true_set = set(random.sample(inputs, k=len(inputs) // 2))

        def f(a, b, c, d):
            return (a, b, c, d) in true_set

    return f

f = random_constant_balanced()
print(f(False, False, False, False))

True


## Problem 2

### Classical Testing for Function Type
Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

In [ ]:
def determine_constant_balanced(f):
    """
    Determine whether a 4-input Boolean function is constant or balanced.

    Assumes f is promised to be either:
    - constant: all 16 outputs are identical, or
    - balanced: exactly 8 True and 8 False outputs.
    """
    true_count = 0
    false_count = 0

    for a, b, c, d in product([False, True], repeat=4):
        result = bool(f(a, b, c, d))

        if result:
            true_count += 1
        else:
            false_count += 1

        # Guaranteed-balanced functions cannot produce more than 8 of one value.
        # If we ever exceed 8, the function must be constant.
        if true_count > 8 or false_count > 8:
            return "constant"

        # Seeing both outputs means the function is not constant; with the promise, it is balanced.
        if true_count > 0 and false_count > 0:
            return "balanced"

    # If all observed outputs were the same, it's constant.
    return "constant"

# Example usage
f = random_constant_balanced()
kind = determine_constant_balanced(f)
print(f"Function type: {kind}")

print("Worst-case classical calls needed for certainty: 9")

Function type: balanced


## Problem 3

### Quantum Oracles
Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

In [ ]:
"""
Quantum oracles for Deutsch's algorithm:
- f(x)=0: identity
- f(x)=1: X on target
- f(x)=x: CNOT
- f(x)=not x: X, CNOT, X
"""

from qiskit import QuantumCircuit


def oracle_f0():
    """f(x)=0: |x, y> -> |x, y>"""
    qc = QuantumCircuit(2, name="Uf0")
    return qc


def oracle_f1():
    """f(x)=1: |x, y> -> |x, y xor 1>"""
    qc = QuantumCircuit(2, name="Uf1")
    qc.x(1)
    return qc


def oracle_fx():
    """f(x)=x: |x, y> -> |x, y xor x>"""
    qc = QuantumCircuit(2, name="Ufx")
    qc.cx(0, 1)
    return qc


def oracle_fnotx():
    """f(x)=not x: |x, y> -> |x, y xor (not x)>"""
    qc = QuantumCircuit(2, name="Ufnotx")
    qc.x(0)
    qc.cx(0, 1)
    qc.x(0)
    return qc


oracles = {
    "f(x)=0": oracle_f0(),
    "f(x)=1": oracle_f1(),
    "f(x)=x": oracle_fx(),
    "f(x)=not x": oracle_fnotx(),
}

for name, oracle in oracles.items():
    print(f"\n{name}")
    print(oracle.draw(output="text"))


f(x)=0
     
q_0: 
     
q_1: 
     

f(x)=1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

f(x)=x
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

f(x)=not x
     ┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     └───┘┌─┴─┐└───┘
q_1: ─────┤ X ├─────
          └───┘     


## Problem 4

### Deutsch's Algorithm with Qiskit
Use Qiskit to design a quantum circuit that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.

## Problem 5

### Scaling to the Deutsch–Jozsa Algorithm
The Deutsch–Jozsa algorithm generalizes Deutsch's approach to functions with several input bits. Use Qiskit to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.